In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-1.5B"
adapter_path = "/kaggle/input/peft-qlora-adapters-for-qwen-1-5b/transformers/default/1"
output_dir = "./merged_model_fp16"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)
model = model.merge_and_unload()

model.save_pretrained(output_dir, safe_serialization=True)
tokenizer.save_pretrained(output_dir)

2026-01-13 06:24:48.502283: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768285488.523201     227 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768285488.534674     227 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768285488.554937     227 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768285488.554959     227 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768285488.554962     227 computation_placer.cc:177] computation placer alr

('./merged_model_fp16/tokenizer_config.json',
 './merged_model_fp16/special_tokens_map.json',
 './merged_model_fp16/chat_template.jinja',
 './merged_model_fp16/vocab.json',
 './merged_model_fp16/merges.txt',
 './merged_model_fp16/added_tokens.json',
 './merged_model_fp16/tokenizer.json')

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "Qwen/Qwen2.5-1.5B"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="cpu"
# )

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model.save_pretrained("./quantized/model-fp16")
# tokenizer.save_pretrained("./quantized/model-fp16")

In [2]:
pip install -U bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_path = "/kaggle/working/merged_model_fp16"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="cpu"
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model.save_pretrained("./quantized/model-int8")
tokenizer.save_pretrained("./quantized/model-int8")


('./quantized/model-int8/tokenizer_config.json',
 './quantized/model-int8/special_tokens_map.json',
 './quantized/model-int8/chat_template.jinja',
 './quantized/model-int8/vocab.json',
 './quantized/model-int8/merges.txt',
 './quantized/model-int8/added_tokens.json',
 './quantized/model-int8/tokenizer.json')

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_path = "/kaggle/working/merged_model_fp16"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="cpu"
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model.save_pretrained("./quantized/model-int4")
tokenizer.save_pretrained("./quantized/model-int4")


('./quantized/model-int4/tokenizer_config.json',
 './quantized/model-int4/special_tokens_map.json',
 './quantized/model-int4/chat_template.jinja',
 './quantized/model-int4/vocab.json',
 './quantized/model-int4/merges.txt',
 './quantized/model-int4/added_tokens.json',
 './quantized/model-int4/tokenizer.json')

In [6]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 75758, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 75758 (delta 120), reused 53 (delta 53), pack-reused 75557 (from 3)
Receiving objects: 100% (75758/75758), 279.95 MiB | 41.30 MiB/s, done.
Resolving deltas: 100% (54959/54959), done.


Below are commands saved in cells for terminal/console use only

!cd llama.cpp
!pip install -r requirements.txt

!python convert_hf_to_gguf.py \/kaggle/working/merged_model_fp16 \--outfile ./quantized/model.gguf \--outtype f16

!mkdir llama.cpp/build && cd llama.cpp/build && cmake .. && cmake --build . --config Release
!cd llama.cpp/build/bin && \./llama-quantize \/kaggle/working/quantized/model.gguf \/kaggle/working/quantized/model-q4_0.gguf \q4_0


!zip -r models_artifacts.zip quantized merged_model_fp16